In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import json
import time
import torch
import torch.nn as nn
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")
V1_DIR = PROJECT_DIR / "Dataset_V1"
FINAL_DIR = V1_DIR / "Final_Method"

FINAL_EVAL_DIR = FINAL_DIR / "Final_Evaluation"
FINAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = FINAL_DIR / "Dataset_V1_splits_FROZEN.csv"
CONFIG_FILE = FINAL_DIR / "FINAL_PIPELINE_CONFIG.json"
MODEL_FILE = FINAL_DIR / "best_unet_model.pth"

df = pd.read_csv(SPLIT_FILE)
test_df = df[df["split"] == "test"].copy()

with open(CONFIG_FILE, "r") as f:
    config = json.load(f)

final_method = config["final_method"]
final_parameters = config["parameters"]

print("FINAL HELD-OUT TEST EVALUATION")


print("\nFinal method:", final_method)
print("Final parameters:", final_parameters)
print("Held-out test samples:", len(test_df))

def load_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def clahe_enhance(img, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=float(clip_limit),
        tileGridSize=tuple(tile_grid)
    )
    l = clahe.apply(l)
    return cv2.cvtColor(
        cv2.merge([l, a, b]),
        cv2.COLOR_LAB2RGB
    )

def gamma_enhance(img, gamma=1.3):
    img_float = img.astype(np.float32) / 255.0
    corrected = np.power(img_float, float(gamma))
    return np.clip(
        corrected * 255,
        0,
        255
    ).astype(np.uint8)

def gray_world(img, strength=1.0):
    img_float = img.astype(np.float32)
    means = img_float.reshape(-1, 3).mean(axis=0)
    gray = means.mean()
    scale = gray / (means + 1e-6)
    corrected = np.clip(
        img_float * scale,
        0,
        255
    )
    result = (
        strength * corrected
        + (1 - strength) * img_float
    )
    return np.clip(
        result,
        0,
        255
    ).astype(np.uint8)

def apply_method(img, method, params):

    method_clean = (
        method
        .lower()
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
    )

    if method_clean == "clahe":
        return clahe_enhance(
            img,
            params["clip_limit"],
            tuple(params["tile_grid"])
        )

    if method_clean == "gamma":
        return gamma_enhance(
            img,
            params["gamma"]
        )

    if method_clean in [
        "grayworld",
        "grayworldwhitebalance"
    ]:
        return gray_world(
            img,
            params["strength"]
        )

    raise ValueError(
        f"Unknown method: {method}"
    )

def edge_metrics(enhanced, target):

    enhanced_gray = cv2.cvtColor(
        enhanced,
        cv2.COLOR_RGB2GRAY
    )

    target_gray = cv2.cvtColor(
        target,
        cv2.COLOR_RGB2GRAY
    )

    enhanced_edges = cv2.Canny(
        enhanced_gray,
        100,
        200
    )

    target_edges = cv2.Canny(
        target_gray,
        100,
        200
    )

    enhanced_binary = enhanced_edges > 0
    target_binary = target_edges > 0

    intersection = np.logical_and(
        enhanced_binary,
        target_binary
    ).sum()

    precision = (
        intersection /
        enhanced_binary.sum()
        if enhanced_binary.sum() > 0
        else 0
    )

    recall = (
        intersection /
        target_binary.sum()
        if target_binary.sum() > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall > 0
        else 0
    )

    preservation = (
        intersection /
        target_binary.sum()
        if target_binary.sum() > 0
        else 0
    )

    return preservation, precision, recall, f1

def calculate_metrics(output, target):

    psnr = peak_signal_noise_ratio(
        target,
        output,
        data_range=255
    )

    ssim = structural_similarity(
        target,
        output,
        channel_axis=2,
        data_range=255
    )

    edge_preservation, edge_precision, edge_recall, edge_f1 = (
        edge_metrics(
            output,
            target
        )
    )

    return {
        "PSNR": psnr,
        "SSIM": ssim,
        "Edge Preservation": edge_preservation,
        "Edge Precision": edge_precision,
        "Edge Recall": edge_recall,
        "Edge F1": edge_f1
    }

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.output = nn.Conv2d(
            32,
            3,
            1
        )

    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool1(e1)
        )

        e3 = self.enc3(
            self.pool2(e2)
        )

        b = self.bottleneck(
            self.pool3(e3)
        )

        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(d3)

        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(d2)

        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(d1)

        return torch.sigmoid(
            self.output(d1)
        )

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

unet = None

if MODEL_FILE.exists():

    unet = UNet().to(device)

    checkpoint = torch.load(
        MODEL_FILE,
        map_location=device
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        unet.load_state_dict(
            checkpoint["model_state_dict"]
        )
    else:
        unet.load_state_dict(
            checkpoint
        )

    unet.eval()

    print("\nU-Net loaded successfully.")
    print("Device:", device)

def prepare_unet_input(img):

    resized = cv2.resize(
        img,
        (224, 224)
    )

    tensor = (
        torch.from_numpy(
            resized.astype(
                np.float32
            ) / 255.0
        )
        .permute(2, 0, 1)
        .unsqueeze(0)
    )

    return tensor.to(device)

results = []

for _, row in test_df.iterrows():

    input_img = load_rgb(
        PROJECT_DIR / row["input"]
    )

    target_img = load_rgb(
        PROJECT_DIR / row["target"]
    )

    start = time.perf_counter()

    baseline_output = input_img.copy()

    baseline_time = (
        time.perf_counter()
        - start
    )

    baseline_metrics = calculate_metrics(
        baseline_output,
        target_img
    )

    results.append({
        "sample_id":
            row["sample_id"],
        "method":
            "Input Baseline",
        "PSNR":
            baseline_metrics["PSNR"],
        "SSIM":
            baseline_metrics["SSIM"],
        "Edge Preservation":
            baseline_metrics["Edge Preservation"],
        "Edge Precision":
            baseline_metrics["Edge Precision"],
        "Edge Recall":
            baseline_metrics["Edge Recall"],
        "Edge F1":
            baseline_metrics["Edge F1"],
        "Processing Time":
            baseline_time
    })

    start = time.perf_counter()

    final_output = apply_method(
        input_img,
        final_method,
        final_parameters
    )

    final_time = (
        time.perf_counter()
        - start
    )

    final_metrics = calculate_metrics(
        final_output,
        target_img
    )

    results.append({
        "sample_id":
            row["sample_id"],
        "method":
            "Final Tuned Method",
        "PSNR":
            final_metrics["PSNR"],
        "SSIM":
            final_metrics["SSIM"],
        "Edge Preservation":
            final_metrics["Edge Preservation"],
        "Edge Precision":
            final_metrics["Edge Precision"],
        "Edge Recall":
            final_metrics["Edge Recall"],
        "Edge F1":
            final_metrics["Edge F1"],
        "Processing Time":
            final_time
    })

    if unet is not None:

        unet_input = prepare_unet_input(
            input_img
        )

        target_resized = cv2.resize(
            target_img,
            (224, 224)
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.no_grad():
            prediction = unet(
                unet_input
            )

        if device.type == "cuda":
            torch.cuda.synchronize()

        unet_time = (
            time.perf_counter()
            - start
        )

        unet_output = (
            prediction
            .squeeze(0)
            .permute(1, 2, 0)
            .cpu()
            .numpy()
            * 255
        )

        unet_output = np.clip(
            unet_output,
            0,
            255
        ).astype(np.uint8)

        unet_metrics = calculate_metrics(
            unet_output,
            target_resized
        )

        results.append({
            "sample_id":
                row["sample_id"],
            "method":
                "U-Net",
            "PSNR":
                unet_metrics["PSNR"],
            "SSIM":
                unet_metrics["SSIM"],
            "Edge Preservation":
                unet_metrics["Edge Preservation"],
            "Edge Precision":
                unet_metrics["Edge Precision"],
            "Edge Recall":
                unet_metrics["Edge Recall"],
            "Edge F1":
                unet_metrics["Edge F1"],
            "Processing Time":
                unet_time
        })

results_df = pd.DataFrame(
    results
)

results_df.to_csv(
    FINAL_EVAL_DIR /
    "final_test_results_per_sample.csv",
    index=False
)

summary = (
    results_df
    .groupby("method")
    [
        [
            "PSNR",
            "SSIM",
            "Edge Preservation",
            "Edge Precision",
            "Edge Recall",
            "Edge F1",
            "Processing Time"
        ]
    ]
    .mean()
    .reset_index()
)

summary.to_csv(
    FINAL_EVAL_DIR /
    "final_test_results_summary.csv",
    index=False
)

baseline = summary[
    summary["method"] == "Input Baseline"
].iloc[0]

final_row = summary[
    summary["method"] == "Final Tuned Method"
].iloc[0]

tradeoff = {
    "final_method": final_method,
    "baseline": "Input Baseline",
    "final_PSNR": float(final_row["PSNR"]),
    "baseline_PSNR": float(baseline["PSNR"]),
    "PSNR_change": float(
        final_row["PSNR"] -
        baseline["PSNR"]
    ),
    "final_SSIM": float(final_row["SSIM"]),
    "baseline_SSIM": float(baseline["SSIM"]),
    "SSIM_change": float(
        final_row["SSIM"] -
        baseline["SSIM"]
    ),
    "final_edge_preservation": float(
        final_row["Edge Preservation"]
    ),
    "baseline_edge_preservation": float(
        baseline["Edge Preservation"]
    ),
    "edge_preservation_change": float(
        final_row["Edge Preservation"] -
        baseline["Edge Preservation"]
    ),
    "final_processing_time_seconds": float(
        final_row["Processing Time"]
    ),
    "baseline_processing_time_seconds": float(
        baseline["Processing Time"]
    )
}

if MODEL_FILE.exists():

    model_size_mb = (
        MODEL_FILE.stat().st_size
        / (1024 ** 2)
    )

else:

    model_size_mb = None

tradeoff["final_method_file_size_MB"] = (
    float(model_size_mb)
    if model_size_mb is not None
    else None
)

if MODEL_FILE.exists():

    tradeoff["unet_model_size_MB"] = float(
        MODEL_FILE.stat().st_size
        / (1024 ** 2)
    )

else:

    tradeoff["unet_model_size_MB"] = None

with open(
    FINAL_EVAL_DIR /
    "quality_complexity_tradeoff.json",
    "w"
) as f:

    json.dump(
        tradeoff,
        f,
        indent=4
    )

final_record = {
    "evaluation_status": "COMPLETED",
    "evaluation_set": "Frozen held-out test set",
    "test_samples": len(test_df),
    "final_method": final_method,
    "final_parameters": final_parameters,
    "test_used_for_method_selection": False,
    "preprocessing_frozen": True,
    "split_frozen": True,
    "evaluation_methodology_frozen": True,
    "model_size_MB": model_size_mb
}

with open(
    FINAL_EVAL_DIR /
    "FINAL_EVALUATION_RECORD.json",
    "w"
) as f:

    json.dump(
        final_record,
        f,
        indent=4
    )

print("FINAL TEST RESULTS")


print(summary)


print("FINAL METHOD VS SIMPLE BASELINE")


print(
    "PSNR change:",
    round(
        tradeoff["PSNR_change"],
        4
    )
)

print(
    "SSIM change:",
    round(
        tradeoff["SSIM_change"],
        4
    )
)

print(
    "Edge Preservation change:",
    round(
        tradeoff["edge_preservation_change"],
        4
    )
)

print(
    "Final processing time:",
    round(
        tradeoff[
            "final_processing_time_seconds"
        ],
        6
    ),
    "seconds/image"
)

if model_size_mb is not None:

    print(
        "Final method file size:",
        round(
            model_size_mb,
            3
        ),
        "MB"
    )

print("\nSaved files:")

for file in sorted(
    FINAL_EVAL_DIR.iterdir()
):

    print(
        " -",
        file.name
    )

print("\nFinal held-out evaluation complete.")
print("No method or parameter was selected using test results.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL HELD-OUT TEST EVALUATION

Final method: Gamma
Final parameters: {'gamma': 0.8}
Held-out test samples: 48

U-Net loaded successfully.
Device: cpu
FINAL TEST RESULTS
               method       PSNR      SSIM  Edge Preservation  Edge Precision  \
0  Final Tuned Method  13.844543  0.654768           0.007723        0.025546   
1      Input Baseline  13.365501  0.654419           0.008117        0.025219   
2               U-Net  14.162797  0.630716           0.032845        0.014136   

   Edge Recall   Edge F1  Processing Time  
0     0.007723  0.010610         0.004036  
1     0.008117  0.010812         0.000045  
2     0.032845  0.017006         0.330070  
FINAL METHOD VS SIMPLE BASELINE
PSNR change: 0.479
SSIM change: 0.0003
Edge Preservation change: -0.0004
Final processing time: 0.004036 seconds/image
Final method file size: 7.358 MB

Saved files:
 -